<a href="https://colab.research.google.com/github/elhamod/IS834_Fall_2026/blob/main/Session%2008%20-%20Combining%20Data%20and%20Intro%20to%20Machine%20Learning/09_Fundamentals_Wrapup_ML1_Intro.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# IS834 — Session 8

Work through this notebook in Colab. The lecture cells are filled in and ready to run; the
**Case** and the **Exercises** are yours to complete — write your code in the `# Your code here` cells and your
reasoning in the `*Provide your answer here*` cells.

Every dataset is read straight from the course repository, so there is nothing to upload or install.

*Adapted from Brock Tibert's IS834 Fall 2025 session 9. Edited for Fall 2026 by Mohannad Elhamod.*

# Combining Data, and an Introduction to Machine Learning

__Learning Goals__:

- Join two tables with `pd.merge`, and choose what happens to rows that have no match (`how=`)
- Audit a join before trusting it with `validate=`
- Frame a business question as a supervised learning problem: features, target, and a train/test split
- Fit a first scikit-learn model and judge it against a do-nothing baseline

## Setup

In [ ]:
# Everything this notebook needs. All of these packages ship with Colab, so nothing needs installing.
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.metrics import accuracy_score

# Every data file this notebook uses is read straight from the course repository.
# There is nothing to upload: DATA is the folder those files live in.
DATA = "https://raw.githubusercontent.com/elhamod/IS834_Fall_2026/main/data/"

## Part 1: Combining data


<img src="https://encrypted-tbn0.gstatic.com/images?q=tbn:ANd9GcQFBXIyLUKfDh1FQV32dYacKqIpW9m-ParnoAJZfyw5n02Zg_Zo6Da4hO8inMBSOSEQYK0&usqp=CAU" width="600">



As we dive into pandas, it's important to call out that the library effectively enables many operations that you would perform in a SQL database.

[Documentation on merge](https://pandas.pydata.org/pandas-docs/stable/reference/api/pandas.merge.html)


### The vocabulary you need first

A **join** (pandas calls it a **merge**) glues two tables together side by side, matching rows using a **key** — a column that means the same thing in both tables, such as a customer id.

The only real decision is *what to do with rows that have no match on the other side*. That is what the `how=` argument controls:

| `how=` | Which rows survive | Use it when |
|---|---|---|
| `"inner"` | only keys found in **both** tables | you want complete records only |
| `"left"` | **every** row of the left table, matched or not | the left table is your population and you are adding detail to it |
| `"right"` | every row of the right table | rare — usually you just swap the tables and use `left` |
| `"outer"` | every row from **both** tables | you are reconciling two lists and want to see what is missing from each |

Unmatched rows are not dropped in a left, right, or outer join — they are kept, and the columns that came from the other table are filled with `NaN` (pandas' "missing").


[A simple webapp tutorial](https://storage.googleapis.com/qst-public/teaching-apps/pandas-merge.html) to help build intuition on how we can join, or blend, data.  It's not always the case that our dataset for an analysis project is simple single-file dataset.


**In-class exercise (10 minutes).** Review the app above to see how joins can alter our data. Then, the tricky part: **replicate the two dataframes from the webapp in this notebook and reproduce the joins yourself, to prove the output.**

> Tip: take screenshots of the app so you have something to check your results against.

The rest of this part works through that exercise.

In [ ]:
# Rebuild the two small tables from the webapp so we can reproduce its joins.
students_data = {
    'student_id': [101, 102, 103, 105],
    'name': ['Alice', 'Bob', 'Charlie', 'Eve'],
    'grade': ['A', 'B', 'A', 'C']
}
students = pd.DataFrame(students_data)

# The emails table.
emails_data = {
    'student_id': [101, 102, 104, 106],
    'email': ['alice@school.edu', 'bob@school.edu', 'david@school.edu', 'frank@school.edu']
}
emails = pd.DataFrame(emails_data)


In [ ]:
# .shape reports (rows, columns) -- write these numbers down before you join anything.
print(students.shape)
print(emails.shape)


In [ ]:
# Look at both tables before joining.
print(students.head(3))
print(emails)


### Inner join — keep only the matches


In [ ]:
# how="inner" keeps a row only when the student_id is present in BOTH tables.
result = pd.merge(
    students,
    emails,
    on='student_id',
    how='outer'
)
result


> Four students went in, four emails went in, **two rows came out**. Charlie (103) and Eve (105) have no email on file, and David (104) and Frank (106) are not enrolled, so all four of those rows were silently discarded.
>
> "Silently" is the word to worry about. pandas did not warn you. If you had 45,000 students you would never have noticed.


### Left join — keep every row of the left table


In [ ]:
# how="left" keeps all four students and fills in an email only where one exists.
left_joined = pd.merge(students, emails, on='student_id', how='left')
left_joined


In [ ]:
# The unmatched students are the ones whose email came back missing.
left_joined[left_joined['email'].isna()]


### Right and outer joins


In [ ]:
# how="right" keeps every email address instead, matched to a student where possible.
pd.merge(students, emails, on='student_id', how='right')


In [ ]:
# how="outer" keeps every row from both sides; indicator=True adds a _merge column saying where each row came from.
outer_joined = pd.merge(students, emails, on='student_id', how='outer', indicator=True)
outer_joined[outer_joined["_merge"] == "left_only"]


In [ ]:
# Counting the indicator column summarises the whole reconciliation in one line.
outer_joined['_merge'].value_counts()


> **Read the `_merge` column.** `both` = matched. `left_only` = a student with no email. `right_only` = an email address with no matching student. Two of each, which is exactly what the webapp showed.
>
> Six rows out of two four-row tables. Nothing was lost, and nothing was invented — but the table is now bigger than either input, which is worth pausing on before you go and compute an average of anything.


### Make pandas validate for you

`validate=` tells `pd.merge` what relationship you *believe* the keys have. If reality disagrees, it raises an error instead of quietly handing you a bigger table — which is what happens when the key repeats in the table you join to, because one row then matches several.

- `"one_to_one"` — one row per key on both sides
- `"one_to_many"` / `"many_to_one"` — one side is a unique lookup
- `"many_to_many"` — no expectation, the default behaviour


In [ ]:
# A second emails table in which student 101 has two addresses on file.
emails_two = pd.DataFrame({
    'student_id': [101, 101, 102],
    'email': ['alice@school.edu', 'alice.home@mail.com', 'bob@school.edu']
})

# A plain left join: four students go in.
pd.merge(students, emails_two, on='student_id', how='left')

In [ ]:
# The same left join, with validate= added. It is meant to fail: try/except prints the error instead of stopping the notebook.
try:
  pd.merge(students, emails_two, on='student_id', how='left', validate='one_to_one')
except Exception as error:
    print("too bad")

**Question 1.** Four students went in and the plain left join quietly returned five rows. What did adding `validate='one_to_one'` change, and what do you do once you get that error?

**Answer**

*Provide your answer here*

### Case: joining the bank marketing data

The same join, on real data: a bank keeps its customer records and its marketing outcomes in two separate files.

1. Load in the __two__ bank datasets from the course repository.
2. Join the two datasets.
3. Look at a correlation matrix of the numeric features.
4. Calculate the average number of outreach interactions (`campaign`) by communication channel (`contact`).  Also include the average `balance`.
5. Calculate the percentage of rows in the dataset that converted as part of the marketing efforts (`y`).

#### Bank Marketing Dataset Overview

| **Attribute** | **Description** | **Type / Values** |
|----------------|-----------------|-------------------|
| **id** | The bank customer's id | String |
| **age** | Age of the client | Numeric |
| **job** | Type of job | Categorical: *admin.*, *unknown*, *unemployed*, *management*, *housemaid*, *entrepreneur*, *student*, *blue-collar*, *self-employed*, *retired*, *technician*, *services* |
| **marital** | Marital status | Categorical: *married*, *divorced*, *single* |
| **education** | Level of education | Categorical: *unknown*, *primary*, *secondary*, *tertiary* |
| **default** | Has credit in default? | Binary: *yes*, *no* |
| **balance** | Average yearly balance (in euros) | Numeric |
| **housing** | Has housing loan? | Binary: *yes*, *no* |
| **loan** | Has personal loan? | Binary: *yes*, *no* |
| **contact** | Contact communication type | Categorical: *unknown*, *telephone*, *cellular* |
| **day** | Last contact day of the month | Numeric |
| **month** | Last contact month of year | Categorical: *jan*, *feb*, ..., *dec* |
| **duration** | Last contact duration (in seconds) | Numeric |
| **campaign** | Number of contacts performed during this campaign (includes last contact) | Numeric |
| **pdays** | Days passed since client was last contacted (-1 = never contacted) | Numeric |
| **previous** | Number of contacts before this campaign | Numeric |
| **poutcome** | Outcome of previous marketing campaign | Categorical: *unknown*, *other*, *failure*, *success* |
| **y** | Client subscribed a term deposit? | Binary: *yes*, *no* |


| **Citation** | Moro et al., 2011 — *Using Data Mining for Bank Direct Marketing: An Application of the CRISP-DM Methodology* ([PDF](http://hdl.handle.net/1822/14838)) |


In [ ]:
# Your code here -- q1: load the two bank files
customers = pd.read_csv(DATA + "bank-customers.csv")
marketing = pd.read_csv(DATA + "bank-marketing.csv")

In [ ]:
# Your code here -- look at both tables before joining (size, columns, a few rows)

In [ ]:
# Your code here -- check that id is a safe key in both tables

In [ ]:
# Your code here -- q2: join the two tables

In [ ]:
# Your code here -- did any customer fail to match?

**Question 1.** Inner and left give the same 45,211 rows here, and no customer came back with a missing `y`. Why is that a result worth printing rather than assuming?

**Answer**

*Provide your answer here*

In [ ]:
# Your code here -- q3: correlation matrix of the numeric columns

**Question 2.** Nothing in that matrix is strongly correlated with anything else. Is that a disappointing result?

**Answer**

*Provide your answer here*

In [ ]:
# Your code here -- q4: average campaign and balance by contact channel

In [ ]:
# Your code here -- q5: percentage of customers who converted

**Question 3.** What share of customers converted, and what stands out in the channel table?

**Answer**

*Provide your answer here*

### Follow up Questions and Exercises

**Question 1.** You join a 10,000-row customer table to a transactions table and get 45,000 rows back. Nothing errored. What happened, and is it a bug?


**Answer**

*Provide your answer here*

**Question 2.** Why does `pd.merge(a, b, on="id", how="left")` sometimes produce columns full of `NaN` that were not missing in either input file?


**Answer**

*Provide your answer here*

**Exercise 1.** The code below is meant to find the students who do not have an email address on file. Run it. Does the result match what you found by hand with the left join above? If not, why not?

In [ ]:
# An attempt at finding the students with no email on file.
merged = pd.merge(students, emails, on="student_id")
missing_email = merged[merged["email"].isna()]
missing_email


**Answer**

*Provide your answer here*

**Exercise 2.** Fix the code above so that it finds the students who really have no email on file.

In [ ]:
# Your code here

**Answer**

*Provide your answer here*

**Exercise 3.** Count how many customers in the joined `bank` table converted (`y == "yes"`) in each contact channel, and turn that into a conversion rate per channel.


In [ ]:
# Your code here

**Answer**

*Provide your answer here*

**Exercise 4.** Build a small table of the ten customers with the highest balance, showing only their id, age, job and balance — and confirm you have not accidentally lost anyone with a tied balance.


In [ ]:
# Your code here

**Answer**

*Provide your answer here*

## Part 2: Introduction to machine learning

![](https://storage.googleapis.com/qst-public/screenshots/pythonarc.jpg)

- Descriptive: Summarize "_what happened_" (ad-hoc analysis, data exploration, hypothesis testing)
- Predictive: Can we determine what "_will happen_"?


### Supervised and unsupervised learning

![](https://assets.extrahop.com/images/blogart/supervised-vs-unsupervised-ml.png)

Supervised:
- We have historical datasets that have recorded outcomes
- We use the past data to train models that can be used to make predictions about future outcomes


Unsupervised (Session 10):
- We don't have labeled data (or choose to ignore it)
- The aim is to find patterns, or groups, in our data.  

![](https://images.datacamp.com/image/upload/v1661171231/Supervised_vs_Unsupervised_aa9a08ac32.png)


### What "supervised" actually means, and when you get to use it

Supervised learning needs one thing that is easy to say and often hard to get: **rows where you already know the answer.**

In the bank data, that column is `y` — for all 45,211 customers we know whether they subscribed, because the campaign already ran. The algorithm's job is to learn the pattern connecting the *inputs* (age, job, balance, channel) to that known *outcome*, well enough to guess the outcome for a customer it has never seen.

The vocabulary, which is worth getting straight now because every tutorial you read assumes it:

- **Features** (`X`) — the input columns, one row per case. Also called predictors, inputs, or independent variables.
- **Target** (`y`) — the single column you are trying to predict. Also called the label, outcome, or dependent variable.
- **Fit / train** — the step where the algorithm looks at `X` and `y` together and works out the pattern.
- **Predict** — applying that fitted pattern to rows it has not seen.

So the honest test of whether supervised learning applies to a business question is: *can I produce a table with one row per case, a column of inputs, and a column recording what actually happened?* If you cannot get that last column, no amount of Python will help — that is the situation Session 10's unsupervised methods are for.


### Why we split the data before fitting anything

![](https://www.sharpsightlabs.com/wp-content/uploads/2022/05/machine-learning-process_train-and-test.png)

![](https://raw.githubusercontent.com/mGalarnyk/Python_Tutorials/df8902b92ce1810834636220d961c985addb2bcd/Sklearn/Train_Test_Split/images/TrainTestProcedure.png)


A model that has seen a row can simply memorise it. Asking it about that same row afterwards tells you nothing about whether it learned a *pattern* — only that it has a good memory.

So we split the rows in two before we fit:

- the **training set** (usually 70–80% of rows) is what the algorithm learns from;
- the **test set** (the rest) is locked away, and only used at the end to see how the model does on cases it genuinely has not seen.

Two arguments in `train_test_split()` are worth understanding rather than copying:

- `random_state=595` fixes the shuffle, so you and your neighbour get the same split and the same numbers. Any number works; the point is that it does not change between runs.
- `stratify=y` keeps the proportion of each outcome the same in both halves. With a rare outcome — like the bank's 11.7% conversions — an unlucky random split could leave you with far too few in one half, and stratifying rules that out.


### Hands-on: predicting a diamond's cut

We are going to use the diamonds dataset that we have seen earlier in the year.  It has some really nice properties that can help us get started with machine learning in Python via Scikit-learn.

**Resources:**

- [scikit-learn project page](https://scikit-learn.org/stable/)
- [Getting Started Guide](https://scikit-learn.org/stable/getting_started.html)
- [Choosing the right estimator](https://scikit-learn.org/stable/machine_learning_map.html) — a flowchart from the data you have to the model to try
- [Glossary of common terms](https://scikit-learn.org/stable/glossary.html) — what `fit`, `transform`, `estimator` and `predict` actually mean
- [Preprocessing with Scikit-learn](https://scikit-learn.org/stable/modules/preprocessing.html)
- [Metrics and Evaluation](https://scikit-learn.org/stable/modules/model_evaluation.html)


In [ ]:
# The file is read straight from the course repository -- nothing to upload.
diamonds = pd.read_csv(DATA + "diamonds.csv")
diamonds.head()


In [ ]:
# rownames is just the row number from the original file, so it carries no information.
diamonds = diamonds.drop(columns="rownames")
diamonds.info()


#### Choosing a question the data can answer

`cut` is the diamond's workmanship grade, and it is assigned by a human grader looking at the stone's proportions. That makes a good first prediction problem: **can the physical measurements alone tell us whether a diamond was graded `Ideal`?**

It is a *classification* problem — we predict a category (here just two: Ideal or not) rather than a number, which would be *regression* (Session 9) — and it has the property you want in a first model: the answer is neither obvious nor hopeless.


In [ ]:
# .value_counts() shows how common each grade is before we simplify anything.
diamonds['cut'].value_counts()


In [ ]:
# A new column: 1 if the diamond was graded Ideal, 0 otherwise.
# .astype(int) turns the True/False into the 1/0 that scikit-learn expects.
diamonds['is_ideal'] = (diamonds['cut'] == 'Ideal').astype(int)

diamonds['is_ideal'].value_counts(normalize=True)

> 40% of diamonds are Ideal, so a lazy model that guessed "not Ideal" every single time would be right 60% of the time. **That 60% is the number any model has to beat to be worth anything.** It is called the baseline, or the majority-class rate, and working it out before you fit takes ten seconds and saves a lot of embarrassment.


In [ ]:
# The model takes columns: a DataFrame of feature columns (X) and one target column (y).
X = diamonds[['carat', 'depth', 'table']]
y = diamonds['is_ideal']
X.head()

In [ ]:
# 80% to train on, 20% held back to test on, keeping the Ideal/not-Ideal mix in both halves.
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=595, stratify=y
)

print("training rows:", X_train.shape[0])
print("test rows:    ", X_test.shape[0])
print("Ideal share, train:", round(y_train.mean(), 3))
print("Ideal share, test: ", round(y_test.mean(), 3))


#### Decision trees: framing for this course

*   **Simple Recipe:** Decision Trees create easy-to-understand, step-by-step flowcharts for making decisions from data. Great for explaining complex insights to anyone.
*   **Key Drivers:** They automatically pinpoint the most important factors influencing an outcome, helping you focus efforts where they matter most.
*   **Predictive Power:** Can be powerful for _forecasting_ what might happen (e.g., customer churn, project success) based on past data patterns.
*   **Actionable Insights:** Converts raw data into clear, explainable, and executable business strategies, extending the work completed in "descriptive" stages of an analytics project.

> This approach is foundational for more advanced techniques that are widely used today (Random Forest, XGBoost) and often the source of winning Kaggle solutions.

**How the algorithm works**

1.  **Start Big:** Imagine all your data as one big pile.
2.  **Ask the Best Question:** The algorithm finds the single best "yes/no" type question (based on your data's features) that splits the pile into two smaller piles, making each new pile as "pure" as possible regarding the outcome you're trying to predict.
3.  **Keep Asking:** It repeats step 2 on each new pile, finding the next best question to ask, continually subdividing the data.
4.  **Reach an Answer:** This process continues until the piles are "pure enough" (most items in a pile share the same outcome) or no more useful questions can be asked. The final, small piles (called "leaves") represent your predictions or classifications.


**Considerations**

1.  How many "piles" or splits do we allow in the tree.  This is referred to as `max_depth`.
1.  How many rows do we require in a pile before it is considered to be broken up into small piles.  `min_samples_split`.
1.  How many rows do we require for a new pile to be created.  `min_samples_leaf`

> Rows are often referred to as __samples__.


![](https://media.geeksforgeeks.org/wp-content/uploads/20250514105137773224/Decision-Tree-.webp)


A bit more applied, using scikit learn.

![](https://scikit-learn.org/stable/_images/iris.svg)

How these rules are constructed (piecewise linear):

![](https://algobeans.com/wp-content/uploads/2016/07/decision-tree-tutorial-animated3.gif)

Decision trees are a very simple, yet powerful algorithm that scales well to larger datasets.  In practice, our trees _might_ look more like below.

![](https://cdn-images-1.medium.com/max/1024/0*SBQNWUA1dDtFsMHv.png)


__Why Decision Trees?__

- Applicable for both Classification and Regression Tasks
- The trees generate rules that are implicitly if-then rules that simply subset the data to make predictions
- Beyond predictive tasks, can highlight patterns in our dataset that are often can help board-room conversations


#### Fitting the first model

Three lines. This is the whole of scikit-learn's interface, and every model in the library works exactly this way:

1. **create** the model, with whatever settings you want;
2. `.fit(X_train, y_train)` — learn from the training rows;
3. `.predict(X_test)` — apply what was learned to the held-out rows.


In [ ]:
# max_depth=3 keeps the tree small enough to read. min_samples_leaf=50 refuses to
# create a rule that rests on fewer than 50 diamonds.
diamond_model = DecisionTreeClassifier(max_depth=3, min_samples_leaf=50, random_state=595)
diamond_model.fit(X_train, y_train)

# Predictions for the rows the model has never seen.
y_pred = diamond_model.predict(X_test)
y_pred[:10]


In [ ]:
# Accuracy on the held-out test rows, next to the baseline of always guessing 'not Ideal'.
baseline = 1 - y_test.mean()

print("model accuracy:  ", round(accuracy_score(y_test, y_pred), 3))
print("baseline (guess 'not Ideal' every time):", round(baseline, 3))


> The model is right about **89%** of the time, against a 60% baseline. Three physical measurements really do carry most of the information a human grader uses.


In [ ]:
# Draw the tree. Every box is a rule; follow the branches down to reach a prediction.
plt.figure(figsize=(20, 8))
plot_tree(
    diamond_model,
    filled=True,
    feature_names=X_train.columns,
    class_names=['Not Ideal', 'Ideal'],
    rounded=True,
    fontsize=9
)
plt.title("Decision Tree - Predict whether a diamond is graded Ideal")
plt.show()


In [ ]:
# feature_importances_ scores how much each input contributed to the splits.
pd.Series(diamond_model.feature_importances_, index=X.columns).sort_values(ascending=False)


**Question 1.** Read the top of the tree, then look at the importances. What is the model's main rule, and what happened to `carat`?

**Answer**

*Provide your answer here*



<img src="https://miro.medium.com/v2/resize:fit:802/1*J_GTKle7phrecheHa-sjYQ.png" alt="Alt text" width="400"/>


### Follow up Questions and Exercises

**Question 1.** Why can we not simply fit the model on all 53,940 diamonds and report how often it was right?


**Answer**

*Provide your answer here*

**Exercise 1.** Refit the diamonds model with `max_depth=2` and with `max_depth=6`, and compare the accuracies. What does the tree's depth buy you?


In [ ]:
# Your code here

**Answer**

*Provide your answer here*

## What comes next

Today's model was judged with one number, accuracy, and it held up because we checked it against a 60% baseline. Two things were deliberately left for Session 9:

- **The bank table you joined in Part 1.** Its 11.7% conversion rate means a model can score 88% accuracy while finding nobody at all. Session 9 fits the same create/fit/predict recipe to it and introduces the metrics — precision, recall, the confusion matrix — that expose that.
- **How deep a tree should grow.** Exercise 1 above kept the train and test scores close together because `min_samples_leaf=50` held the tree back. Session 9 removes that guardrail and shows what overfitting looks like.